In [ ]:
import numpy as np
import pandas as pd
from scapy.all import rdpcap, sniff, TCPSession

import struct
from collections import Counter
from collections import defaultdict

import zlib


瞎统计一下，发现这个包里全是TCP，并且发送端口全是 5261 的

In [ ]:
packets = sniff(offline="../data/output.pcap", session=TCPSession)
print(len(packets))

src_counter = Counter()  # 源IP（发包方）
dst_counter = Counter()  # 目标IP（收包方）

for pkt in packets:
    if pkt.haslayer("IP"):
        src_counter[f"{pkt['IP'].src}:{pkt['TCP'].sport}"] += 1
        dst_counter[f"{pkt['IP'].dst}:{pkt['TCP'].dport}"] += 1

print("=== 源 IP:Port 分布 ===")
for endpoint, count in src_counter.most_common():
    print(f"  {endpoint}: {count} 个包")

print("\n=== 目标 IP:Port 分布 ===")
for endpoint, count in dst_counter.most_common():
    print(f"  {endpoint}: {count} 个包")

In [ ]:
MAGIC = 0x0004C453

def parse_frames(stream_data, stream_key):
    frames = []
    errors = []
    offset = 0
    data = stream_data
    MAGIC_BYTES = MAGIC.to_bytes(4, "big")  # 转成bytes方便搜索

    while offset < len(data):
        # 剩余不够一个头
        if offset + 40 > len(data):
            break

        header = data[offset:offset+40]
        magic, total_len, typ, subtype, seq, zero1, ts, flag1, compress, flag2 = struct.unpack(">10I", header)

        # 魔数不对，往后搜索下一个魔数
        if magic != MAGIC:
            next_pos = data.find(MAGIC_BYTES, offset + 1)
            if next_pos == -1:
                errors.append(f"[{stream_key}] offset={offset} 魔数不对且后续找不到魔数，放弃")
                break
            else:
                errors.append(f"[{stream_key}] offset={offset} 魔数不对，跳到下一个魔数 offset={next_pos}")
                offset = next_pos
                continue

        # 帧长度越界，同样往后搜
        if offset + total_len > len(data):
            next_pos = data.find(MAGIC_BYTES, offset + 1)
            if next_pos == -1:
                errors.append(f"[{stream_key}] offset={offset} 帧长度越界，放弃")
                break
            else:
                errors.append(f"[{stream_key}] offset={offset} 帧长度越界，跳到下一个魔数 offset={next_pos}")
                offset = next_pos
                continue

        body = data[offset+40 : offset+total_len]
        frames.append({
            "offset":    offset,
            "total_len": total_len,
            "type":      typ,
            "subtype":   subtype,
            "seq":       seq,
            "ts":        ts,
            "flag1":     flag1,
            "compress":  compress,
            "flag2":     flag2,
            "body":      body,
        })

        offset += total_len

    return frames, errors


def verify_seq(frames, stream_key):
    """验证每个(type, subtype)的seq是否严格+1"""
    seq_tracker = {}  # (type, subtype) -> 上一个seq
    seq_errors = []

    for f in frames:
        k = (f["type"], f["subtype"])
        if k in seq_tracker:
            expected = seq_tracker[k] + 1
            if f["seq"] != expected:
                seq_errors.append(
                    f"[{stream_key}] type={f['type']} sub={f['subtype']} "
                    f"seq跳变: 期望{expected} 实际{f['seq']}"
                )
        seq_tracker[k] = f["seq"]

    return seq_errors


def verify_compress(frames, stream_key):
    """验证compress字段和body内容是否一致"""
    compress_errors = []

    for f in frames:
        body = f["body"]
        if f["compress"] == 1:
            if not body.startswith(b"PK\x03\x04"):
                compress_errors.append(
                    f"[{stream_key}] offset={f['offset']} compress=1 但body不是ZIP: {body[:4].hex()}"
                )
        elif f["compress"] == 0:
            if body.startswith(b"PK\x03\x04"):
                compress_errors.append(
                    f"[{stream_key}] offset={f['offset']} compress=0 但body像ZIP"
                )

    return compress_errors

streams = defaultdict(bytes)
for pkt in packets:
    payload = bytes(pkt["TCP"].payload)
    key = (pkt["IP"].src, pkt["TCP"].sport,
            pkt["IP"].dst, pkt["TCP"].dport)
    streams[key] += payload
# 逐流解析+验证
all_frames = []
for key, data in streams.items():
    stream_key = f"{key[0]}:{key[1]}->{key[2]}:{key[3]}"
    print(f"\n{'='*60}")
    print(f"流: {stream_key}  总字节: {len(data)}")

    frames, parse_errors = parse_frames(data, stream_key)
    seq_errors = verify_seq(frames, stream_key)
    compress_errors = verify_compress(frames, stream_key)

    print(f"  解析帧数: {len(frames)}")
    print(f"  解析错误: {len(parse_errors)}")
    print(f"  seq错误:  {len(seq_errors)}")
    print(f"  压缩错误: {len(compress_errors)}")

    for e in parse_errors + seq_errors + compress_errors:
        print(f"  ⚠️  {e}")

    all_frames.extend(frames)

print(f"\n{'='*60}")
print(f"全部流合计帧数: {len(all_frames)}")

In [ ]:
tmp = streams[('1.4.2.53', 5261, '1.6.86.11', 58398)]
print(tmp[26313: 27287])
print(tmp[1212780: 1212929])

In [ ]:
all_frames = []
for key, data in streams.items():
    stream_key = f"{key[0]}:{key[1]}->{key[2]}:{key[3]}"
    print(f"\n{'='*60}")
    print(f"流: {stream_key}  总字节: {len(data)}")

    frames, parse_errors = parse_frames(data, stream_key)
    seq_errors = verify_seq(frames, stream_key)
    compress_errors = verify_compress(frames, stream_key)

    all_frames.extend(frames)

In [ ]:
type_cnt = defaultdict(int)
for f in all_frames:
    type_cnt[(f["type"], f["subtype"])] += 1

type_cnt

In [ ]:
splited_frames = defaultdict(list)
for f in all_frames:
    splited_frames[(f["type"], f["subtype"])].append(f)

In [ ]:
frames9_5803 = splited_frames[(9, 5803)]

for f in frames9_5803:
    if f["compress"] == 0:
        print(f["body"])

In [ ]:

# bytes -> 内存文件 -> zipfile解析
for i, f in enumerate(frames9_5803):
    if f["compress"] == 1:
        body = f["body"]
        # 解析ZIP Local File Header，找到数据起始位置
        # 偏移26: name_len (2字节小端)
        # 偏移28: extra_len (2字节小端)
        name_len  = struct.unpack_from("<H", body, 26)[0]
        extra_len = struct.unpack_from("<H", body, 28)[0]
        start = 30 + name_len + extra_len

        # 取裸deflate数据
        raw = body[start:]

        # 用 wbits=-15 解压，对应C++的 -MAX_WBITS
        print(zlib.decompress(raw, wbits=-15))
    

In [ ]:
def decode_unit(data: bytes, pos: int) -> tuple[int, bytes, int]:
    start = pos
    origin = b""
    while pos < len(data):
        b = data[pos]
        pos += 1
        origin += bytes([b])
        if b & 0x80:
            break
        if len(origin) > 9:
            raise ValueError(f"偏移 {start}: token 过长")
    return origin, pos - start

def unit2num(data: bytes):
    value = 0
    for b in data:
        value = (value << 7) | (b & 0x7f)
    return value

def unit2str(data: bytes):
    code = [b & 0x7f for b in data]
    return bytes(code).decode('ascii')


all_units = []
for f in frames9_5803:
    if f["compress"] == 0:
        body = f["body"]

        pos = 0
        while pos < len(body):
            origin, consumed = decode_unit(body, pos)
            pos += consumed
            all_units.append(origin)

In [ ]:
def split_by_pattern(lst, pattern):
    # 1. 找出所有 pattern 的位置
    pos = [i for i, v in enumerate(lst) if v == pattern]
    if not pos:
        return [lst]

    result = []
    for idx, p in enumerate(pos):
        # 起始：pattern 的前一个元素（若 p==0 则从 0 开始）
        start = p - 1 if p > 0 else 0
        # 结束：下一个 pattern 的前一个元素的前一个位置，或列表末尾
        if idx < len(pos) - 1:
            end = pos[idx + 1] - 2
        else:
            end = len(lst) - 1

        if start <= end:          # 避免空切片（如连续 pattern 时可能出现）
            result.append(lst[start:end+1])
    return result

# 测试
pattern = b'-\xab'

all_ticks = split_by_pattern(all_units, pattern)

pmap_cnt = defaultdict(int)
for tick in all_ticks:
    pmap_cnt[tick[0]] += 1

for k, v in pmap_cnt.items():
    print(f'{k}, {k[0]:02b} {k[1]:02b}, {v}')

In [ ]:
for tick in all_ticks:
    if tick[0] == b'K\xfc':
        print(len(tick), tick)

In [ ]:
def check(data):
    print(f'{data.hex()} || {unit2num(data)} || {unit2str(data)}')

raw = [b'K\xfc', b'-\xab', b'60088\xb4', b'\xd4', b'\x03 k\xe1', b'\x03\x1d \x8b', b'\x00i\xd1', b'\x06\r\xa1', b'\x00@;z\x81', b'\xc2']
for i in raw:
    check(i)